# Inverse Ising via Pseudo-Likelihood (Phase 2)

Direct fit of the binary-spin Ising model $(J, h)$ from team-level tournament data scraped via `tournament_ingest`, replacing the Gaussian / precision-matrix approximation used in `inverse_ising.ipynb`. The phase 1 fit recovers synergy structure cleanly but systematically under-magnitudes strong negative couplings (the mutual-exclusion limitation documented in `CLAUDE.md`). Pseudo-likelihood drops the Gaussian assumption entirely: each spin's full conditional becomes a per-Pokémon logistic regression on the other $V-1$ spins.

Spin convention is $\{0, 1\}$ throughout (consistent with phase 1). The energy is $H(s) = -h \cdot s - \tfrac{1}{2} s^\top J s$, so

$$P(s_i = 1 \mid s_{-i}) = \sigma\!\left(h_i + \sum_{j \neq i} J_{ij}\, s_j\right),$$

which is exactly logistic regression of column $i$ on the other $V-1$ columns of the binary team matrix. Symmetrize $J$ post-hoc since the $V$ regressions don't share coefficients.

Headline claims to test (deferred Phase 1 vs Phase 2 scatter is a separate follow-up):

- Top $+J$ pairs match phase 1's archetype cores (sand, rain, TR cores).
- Top $-J$ pairs include slot-conflict pairs *not* tied to held-item forme distinctions (Pelipper / Politoed for rain, Farigiraf / Oranguru for TR support, etc.). Mega-conflict pairs are invisible by construction — Limitless doesn't differentiate formes (see CLAUDE.md).
- PL $-J$ magnitudes should be visibly larger than phase 1's, since the Gaussian approximation shrinks negatives.

In [ ]:
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

from k2dex import tournament_ingest
from k2dex.constants import SPECIES_LR_C
from k2dex.models import fit_pl_ising

tournaments = tournament_ingest.load_cached_tournaments()
teams = tournament_ingest.all_teams(tournaments)
print(f"tournaments: {len(tournaments)}")
print(f"teams:       {len(teams)}")

## Vocab + binary team matrix

Vocab is built directly from the ingested teams — no cross-source name normalization needed since everything downstream consumes only Limitless data. The min-team-count cutoff drops Pokémon too rare for their per-spin regression to be informative (even with L2, a mon present in <5 teams has too few positive examples to learn meaningfully). The team matrix `X` is `N × V` boolean: row = team, column = Pokémon.

In [ ]:
MIN_TEAM_COUNT = 5

counts = Counter(name for team in teams for name in team)
vocab = sorted(name for name, c in counts.items() if c >= MIN_TEAM_COUNT)
idx = {name: i for i, name in enumerate(vocab)}
V = len(vocab)
print(f"vocab size: {V} (>= {MIN_TEAM_COUNT} teams)")

X = np.zeros((len(teams), V), dtype=bool)
for i, team in enumerate(teams):
    for name in team:
        j = idx.get(name)
        if j is not None:
            X[i, j] = True

team_sizes = X.sum(axis=1)
print(
    f"team size after vocab filter: min={team_sizes.min()}, "
    f"median={np.median(team_sizes):.0f}, max={team_sizes.max()}, "
    f"mean={team_sizes.mean():.2f}"
)

## Empirical marginals

Top-20 most-used Pokémon should look like a recognizable VGC Reg M-A metagame. If this list looks wrong, something upstream is broken (wrong regulation filter in `ingest`, parsing bug, etc.) — diagnose before continuing.

In [ ]:
m = X.mean(axis=0)
order = np.argsort(-m)
print(f"{'rank':>4}  {'pokemon':<25}  {'usage':>7}  {'count':>5}")
for r, j in enumerate(order[:20]):
    print(f"{r+1:>4}  {vocab[j]:<25}  {m[j]:>7.1%}  {int(X[:, j].sum()):>5}")

## Pseudo-likelihood fit

Pseudo-likelihood factorizes the joint over the team-spin vector into a product of full conditionals:

$$\mathcal{L}(J, h) = \prod_i \prod_n P\!\left(s_{n,i} \mid s_{n,-i}, h_i, J_{i, :}\right).$$

Because the Ising conditional is $P(s_i = 1 \mid s_{-i}) = \sigma\!\left(h_i + \sum_j J_{ij} s_j\right)$, each per-spin term is *literally* a logistic regression with intercept $h_i$ and feature coefficients $J_{i, :}$. So fitting reduces to $V$ **independent** logistic regressions — one per Pokémon. For spin $i$ we:

1. Set target `y = X[:, i]` (length-$N$ binary: is mon $i$ on each team?).
2. Set features `Xi = X[:, ~i]` (the other $V-1$ binary columns).
3. Fit logistic regression on `(Xi, y)`. The fitted intercept becomes $\hat h_i$ and the coefficient vector becomes the row $\hat J_{i, :}$.

**`J_asym` and post-hoc symmetrization.** Regression $i$ produces $\hat J_{ij}$ and regression $j$ separately produces $\hat J_{ji}$, from different optimizations on different targets. Nothing in the PL objective constrains them to be equal. In an Ising model the true $J$ is symmetric (the energy uses the bilinear form $\tfrac{1}{2} s^\top J s$), so $\hat J_{ij} \approx \hat J_{ji}$ asymptotically — but in finite samples they differ due to estimation noise and any model misspecification. We keep the asymmetric estimate as `J_asym` for the symmetry diagnostic in the next cell, then take $J = \tfrac{1}{2}(J_\text{asym} + J_\text{asym}^\top)$ — the unique symmetric matrix nearest `J_asym` in Frobenius distance.

**Regularization (L2, `C = 1.0`, lbfgs solver).** sklearn's `LogisticRegression` maximizes regularized log-likelihood

$$\sum_n \log P(y_n \mid x_n, \beta) \;-\; \tfrac{1}{2C} \|\beta\|_2^2.$$

L2 shrinks coefficients toward zero, preventing accidental blowup on weakly-supported pairs (two mons that rarely co-occur producing huge coefficients from tiny correlations). `C` is the *inverse* regularization strength: large `C` = weak penalty, small `C` = strong penalty. `C = 1.0` is sklearn's default and a moderate amount appropriate to our $N \sim 1500$, $V \sim$ few hundred regime. `lbfgs` is the optimizer — a quasi-Newton method, fast and accurate at this scale. Both are knobs worth sweeping if the symmetry diagnostic or coefficient distribution looks pathological.

In [ ]:
X_int = X.astype(np.int32)
J, h = fit_pl_ising(X_int, C=SPECIES_LR_C)

iu = np.triu_indices(V, k=1)
print(f"h range: [{h.min():+.3f}, {h.max():+.3f}]")
print(f"J off-diagonal range: [{J[iu].min():+.3f}, {J[iu].max():+.3f}]")
print(f"|J| off-diagonal median: {np.median(np.abs(J[iu])):.3f}")

## Symmetry diagnostic

$J_{ij}$ and $J_{ji}$ are fit as independent coefficients (in different regressions), but for a valid Ising model they should be equal. Scatter is a direct goodness-of-fit signal: tight clustering on $y = x$ means the model agrees with itself; scatter off-diagonal means the parameters are statistically uncertain, the model is misspecified, or both. A correlation $> 0.9$ is what we want here.

In [ ]:
off_diag = ~np.eye(V, dtype=bool)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(J_asym[off_diag], J_asym.T[off_diag], s=2, alpha=0.3)
lim = max(np.abs(J_asym).max(), np.abs(J_asym.T).max())
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.5)
ax.set_xlabel(r"$J_{ij}$ (regression for $i$)")
ax.set_ylabel(r"$J_{ji}$ (regression for $j$)")
ax.set_aspect("equal")
ax.set_title("PL symmetry (perfect fit lies on y = x)")
plt.tight_layout()
plt.show()

print(f"corr(J_ij, J_ji): {np.corrcoef(J_asym[iu], J_asym.T[iu])[0, 1]:.3f}")

## Top +J pairs (synergies / archetype cores)

Should be dominated by recognizable cores: sand (Tyranitar / Excadrill), rain (Pelipper / wave-rider), TR (Hatterene / Indeedee or Farigiraf / restricted attacker), and so on. If this list is unrecognizable, the fit is broken.

In [ ]:
def top_pairs(J, vocab, n=25, sign=+1):
    M = J if sign > 0 else -J
    iu = np.triu_indices(M.shape[0], k=1)
    vals = M[iu]
    order = np.argsort(-vals)[:n]
    return [(vocab[iu[0][k]], vocab[iu[1][k]], sign * vals[k]) for k in order]


print(f"{'rank':>4}  {'pokemon A':<25} {'pokemon B':<25} {'J':>8}")
for r, (a, b, j) in enumerate(top_pairs(J, vocab, n=25, sign=+1)):
    print(f"{r+1:>4}  {a:<25} {b:<25} {j:>+8.3f}")

## Top −J pairs (role substitutes / slot competitors)

These are the pairs the Gaussian phase 1 fit systematically under-magnitudes. Headline names to watch:

- Pelipper / Politoed (rain setters)
- Hippowdon / Tyranitar (sand setters)
- Farigiraf / Oranguru (TR support)
- Single-strike vs Rapid Strike Urshifu (if both clear vocab cutoff)

Held-item forme conflicts (Charizard-Y vs Charizard-X, Diancie vs Diancie-Mega, etc.) are **invisible** here because Limitless collapses formes to base names — see CLAUDE.md.

In [ ]:
print(f"{'rank':>4}  {'pokemon A':<25} {'pokemon B':<25} {'J':>8}")
for r, (a, b, j) in enumerate(top_pairs(J, vocab, n=25, sign=-1)):
    print(f"{r+1:>4}  {a:<25} {b:<25} {j:>+8.3f}")

## Network panels (sign-split)

Top 60 most-used Pokémon laid out by force-directed (Kamada–Kawai) layout, with edges drawn where coupling magnitude clears a quantile threshold: $+J$ at $q = 0.90$ (top 10%) and $-J$ at $q = 0.95$ (top 5%). Tighter than Phase 1's $q = 0.75$ for $+J$ because PL produces a wider positive-J distribution — at the looser threshold the +J panel becomes too edge-dense for a force-directed layout to spatially separate the archetypes. Node size scales with empirical usage. Isolated nodes per-panel are dropped to keep layouts readable.

Mirrors the same panels in `inverse_ising.ipynb` for direct visual comparison. The $-J$ panel is where the largest qualitative difference is expected: Phase 1's Gaussian fit produced a sparse, hard-to-read $-J$ graph because of the negative-coupling under-magnitude. PL should make the negative side denser and structurally clearer, even though held-item forme conflicts are invisible by construction.

In [ ]:
import networkx as nx

TOP_N = 60
top_idx = np.argsort(-m)[:TOP_N]
top_names = [vocab[i] for i in top_idx]
J_sub = J[np.ix_(top_idx, top_idx)]
m_sub = m[top_idx]

iu_sub = np.triu_indices(TOP_N, k=1)
J_off = J_sub[iu_sub]
pos_mag = J_off[J_off > 0]
neg_mag = -J_off[J_off < 0]
thr_pos = float(np.quantile(pos_mag, 0.85)) if len(pos_mag) else np.inf
thr_neg = float(np.quantile(neg_mag, 0.95)) if len(neg_mag) else np.inf


def build_panel(J_sub, names, usage, threshold, sign):
    G = nx.Graph()
    for n, u in zip(names, usage):
        G.add_node(n, usage=float(u))
    for i, j in zip(*np.triu_indices(len(names), k=1)):
        v = J_sub[i, j]
        if sign * v >= threshold:
            G.add_edge(names[i], names[j], weight=abs(float(v)))
    G.remove_nodes_from([n for n in list(G.nodes) if G.degree(n) == 0])
    return G


G_pos = build_panel(J_sub, top_names, m_sub, thr_pos, +1)
G_neg = build_panel(J_sub, top_names, m_sub, thr_neg, -1)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
panels = [
    (axes[0], G_pos, f"+J  q=0.90  thr={thr_pos:+.2f}  |V|={G_pos.number_of_nodes()}  |E|={G_pos.number_of_edges()}", "tab:blue"),
    (axes[1], G_neg, f"−J  q=0.95  thr={thr_neg:+.2f}  |V|={G_neg.number_of_nodes()}  |E|={G_neg.number_of_edges()}", "tab:red"),
]
for ax, G, title, edge_color in panels:
    if G.number_of_nodes() == 0:
        ax.set_title(title + "  (empty)")
        ax.axis("off")
        continue
    pos = nx.kamada_kawai_layout(G)
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_w = max(weights, default=1.0)
    edge_widths = [3.0 * w / max_w for w in weights]
    node_sizes = [80 + 600 * G.nodes[n]["usage"] / m_sub.max() for n in G.nodes]
    nx.draw_networkx_edges(G, pos, alpha=0.5, width=edge_widths, edge_color=edge_color, ax=ax)
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color="lightgray", edgecolors="black", linewidths=0.5, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Per-query synergies + substitutes

Same query list as the phase 1 notebook for direct side-by-side comparison. Top-8 positive (synergies) and top-8 negative (substitutes) for each.

In [ ]:
QUERIES = ["Hatterene", "Tyranitar", "Pelipper", "Sneasler", "Sinistcha"]

for q in QUERIES:
    if q not in idx:
        print(f"\n!! {q} not in vocab (below cutoff or named differently in Limitless)")
        continue
    i = idx[q]
    j_row = J[i].copy()
    pos = np.argsort(-j_row)[:8]
    neg = np.argsort(j_row)[:8]
    print(f"\n=== {q} ===")
    print(f"  +J (synergies):")
    for k in pos:
        print(f"    {vocab[k]:<25} {j_row[k]:+7.3f}")
    print(f"  -J (substitutes):")
    for k in neg:
        print(f"    {vocab[k]:<25} {j_row[k]:+7.3f}")